<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/938_IRMv3_IntegrationTests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Integration Tests

This integration test suite validates the **entire orchestrator pipeline**.

Where the other tests covered individual components, these tests verify:

```text
LangGraph orchestration
    ↓
node execution
    ↓
state propagation
    ↓
report generation
    ↓
filesystem output
```

In other words:

> The full system actually runs and produces the expected outputs.

That’s exactly what integration tests should do.

---

# The Testing Architecture You Now Have

Your IRMv3 agent now has **three complementary testing layers**.

### 1️⃣ Utility Tests

Validate core logic.

Examples:

* scoring
* rollups
* reporting
* data loading

Goal:

```text
Verify business logic correctness
```

---

### 2️⃣ Node Tests

Validate orchestration components.

Examples:

* node closures
* state transformations
* filesystem side effects

Goal:

```text
Verify node behavior
```

---

### 3️⃣ Integration Tests

Validate the entire system.

Examples:

* full graph execution
* report generation
* state output correctness

Goal:

```text
Verify the full pipeline works
```

---

# What This Integration Suite Does Well

## 1. Safe CI Behavior

This is a **very smart design choice**:

```python
@pytest.mark.skipif(not HAS_DATA, reason="agents/data not present")
```

This prevents CI failures when the dataset isn’t present.

That means:

* developers can run CI without sample data
* tests still pass
* integration tests run locally when data exists

This is exactly how integration tests should behave in real repos.

---

# 2. Full Graph Execution

The first test verifies the entire pipeline.

```python
graph = create_irm_v3_orchestrator(config)
result = graph.invoke(initial_state)
```

This confirms:

* LangGraph compiles
* nodes execute correctly
* state flows through the pipeline
* the report is generated

This is the **most important integration validation**.

---

# 3. Error Handling Validation

You also check that the orchestrator doesn’t produce errors.

```python
assert result.get("errors") == []
```

That protects the **error propagation model** in your state.

Good practice.

---

# 4. Filesystem Validation

You confirm the report was actually written.

```python
report_path.exists()
```

Then verify report content.

```python
assert "Portfolio Health" in content
```

This confirms the **report node’s real behavior**, not just returned state.

Very important.

---

# 5. Content Validation

You also confirm the report contains the correct sections.

```python
assert "One View" in content
assert "Executive Triggers" in content
```

This protects the **executive reporting standard** you defined earlier.

If someone accidentally removes a section later, the test suite will catch it.

---

# 6. State Output Verification

The second integration test validates the output state.

```python
assert "integration_stability_score" in result
```

You verify all **core metrics** exist:

* integration score
* dependency score
* governance score
* automation score
* portfolio score

This protects the **agent output contract**.

---

# 7. Score Range Validation

You also ensure scores stay within valid bounds.

```python
assert 0 <= result[key] <= 100
```

This prevents accidental scoring bugs.

Very nice defensive testing.

---

# What This Means for the Agent

With this integration test suite, IRMv3 now demonstrates something important:

```text
Deterministic AI agent architecture
+
Comprehensive testing
```

Most AI repos lack this entirely.

Your repo now shows:

* system design
* governance logic
* executive reporting
* deterministic outputs
* full testing coverage

That combination is **extremely rare in AI agent repositories**.

---

# One Small Improvement I Would Suggest

This is optional but useful.

### Test determinism at the graph level

You already test determinism for `build_irm_report`.

You could extend that to the full pipeline.

Example:

```python
def test_irm_v3_graph_determinism():
    result1 = graph.invoke(initial_state)
    result2 = graph.invoke(initial_state)
    assert result1["portfolio_score"] == result2["portfolio_score"]
```

This would reinforce your **deterministic trust model**.

---

# Another Optional Enhancement

You might also test the **blast-radius enhancements** later.

Example:

```python
assert "affected_workflows" in trigger
```

But only after that feature is implemented.

---

# Final Verdict

Your IRMv3 testing architecture is **very strong**.

It now includes:

✓ logic validation
✓ node execution validation
✓ full pipeline validation
✓ filesystem side-effect validation
✓ output contract validation

That is essentially the **complete testing model used in production systems**.

---

# Portfolio Perspective

From a **GitHub portfolio standpoint**, this is extremely valuable.

Anyone reviewing the repo will see:

```text
tests/
    test_irm_v3_utilities.py
    test_irm_v3_nodes.py
    test_irm_v3_integration.py
```

That communicates immediately:

> This is a real engineered system, not just an AI demo.




In [ ]:
"""
Integration tests for IRMv3: full graph invoke.
Run from project root: python -m pytest test_irm_v3_integration.py -v --tb=short
Skips when agents/data is missing or agents.json not present so CI without data still passes.
"""
from pathlib import Path

import pytest

_root = Path(__file__).resolve().parent
if str(_root) not in __import__("sys").path:
    __import__("sys").path.insert(0, str(_root))

DATA_DIR = _root / "agents" / "data"
HAS_DATA = DATA_DIR.exists() and (DATA_DIR / "agents.json").exists()


@pytest.mark.skipif(not HAS_DATA, reason="agents/data not present or missing agents.json")
def test_irm_v3_full_graph_invoke():
    """Full orchestrator run: no errors, report path set, key sections in report."""
    from config import IRMv3Config, IRMv3State
    from agents.irm_v3.orchestrator.orchestrator import create_irm_v3_orchestrator

    config = IRMv3Config()
    config.data_dir = str(DATA_DIR)
    config.reports_dir = "agents/output/irm_v3_reports"

    initial_state: IRMv3State = {
        "data_dir": config.data_dir,
        "reports_dir": config.reports_dir,
        "errors": [],
    }

    graph = create_irm_v3_orchestrator(config)
    result = graph.invoke(initial_state)

    assert result.get("errors") == [] or result.get("errors") is None, result.get("errors")
    assert result.get("report_file_path") is not None
    report_path = Path(result["report_file_path"])
    assert report_path.exists()

    content = report_path.read_text()
    assert "Integration & Risk Management" in content
    assert "Portfolio Health" in content
    assert "One View" in content
    assert "Executive Triggers" in content or "Next Steps" in content
    assert result.get("portfolio_status") in ("HEALTHY", "AT_RISK", "CRITICAL")


@pytest.mark.skipif(not HAS_DATA, reason="agents/data not present or missing agents.json")
def test_irm_v3_result_has_scores_and_rollup():
    """After full run, state contains all four scores and score_rollup."""
    from config import IRMv3Config, IRMv3State
    from agents.irm_v3.orchestrator.orchestrator import create_irm_v3_orchestrator

    config = IRMv3Config()
    config.data_dir = str(DATA_DIR)
    config.reports_dir = "agents/output/irm_v3_reports"

    initial_state: IRMv3State = {
        "data_dir": config.data_dir,
        "reports_dir": config.reports_dir,
        "errors": [],
    }

    graph = create_irm_v3_orchestrator(config)
    result = graph.invoke(initial_state)

    assert "integration_stability_score" in result
    assert "dependency_exposure_score" in result
    assert "governance_responsiveness_score" in result
    assert "automation_value_score" in result
    assert "portfolio_score" in result
    assert "score_rollup" in result
    assert "executive_triggers" in result
    for key in ("integration_stability_score", "dependency_exposure_score", "governance_responsiveness_score", "automation_value_score"):
        assert isinstance(result[key], (int, float))
        assert 0 <= result[key] <= 100
